# Privacy and Governance Analysis

This notebook has the intention to analyse the risks regarding privacy on the start-ups credit aplications dataset.

The goals are to identify personal data (PII), evaluate possible risks of not comply with GDPR, show pseudoanonimaztion of the data and also suggest some recommendations regarding data governance inside the organization.

This analysis was performed on the dataset that was already cleaned and prepared by the data team. It will be possile to focus on privacy and governance, preventing techinical mistkes related to data quality.

## 0. Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
import hashlib

df = pd.read_csv('clean_dataset_view.csv')
df.head()

,_id,spending_behavior,processing_timestamp,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,loan_purpose,interest_rate,approved_amount,notes
0,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,female,1986-05-27,90230,102000.0,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR
1,app_002,"[{'category': 'Education', 'amount': 533}]",NaN,Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,male,1999-08-01,10020,41000.0,5,0.36,18200,False,algorithm_risk_score,NaN,NaN,NaN,NaN
2,app_003,"[{'category': 'Healthcare', 'amount': 450}]",NaN,Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,female,1982-08-24,90213,65000.0,74,0.43,7090,True,NaN,NaN,3.4,76000.0,NaN
3,app_004,"[{'category': 'Transportation', 'amount': 329}...",NaN,Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,female,02/28/1995,90217,69000.0,9,0.41,10327,False,high_dti_ratio,NaN,NaN,NaN,NaN
4,app_005,"[{'category': 'Insurance', 'amount': 585}]",NaN,Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,female,1960/06/19,90296,39000.0,76,0.06,15011,False,algorithm_risk_score,NaN,NaN,NaN,NaN


In [5]:
df.columns
columns_table = df.columns.to_frame(index=False, name='Columns')
columns_table

,Columns
0,_id
1,spending_behavior
2,processing_timestamp
3,full_name
4,email
5,ssn
6,ip_address
7,gender
8,date_of_birth
9,zip_code


## 1. Personal Data (PII) & Classification

According to the GDPR, personal data refers to information that allows the identification of an individual, either directly or indirectly. For that reason, it should be treated with special security and privacy measures. In this dataset, several potential fields were identified as Personally Identifiable Information (PII).

After analysing the columns of the dataset, it is possible to conclude that the following columns are considered personal data. We classify each field by PII type and risk level.
- **Direct identifiers** are attributes that can uniquely identify an individual on their own, such as name, email or SSN.
- **Quasi-identifiers** do not identify a person individually but may allow identification when combined with other attributes, such as date of birth, ZIP code or gender.
- **Technical identifier** contains information that might identify an individual or its devices.
- **Free text risk** might contain personal important information hidden. So we can consider it hidden risk.

In [7]:
pii_classification = pd.DataFrame([
    # Direct identifiers — uniquely identify a person on their own
    {'column': '_id',                   'pii_type': 'Direct identifier',    'gdpr_category': 'Personal data',             'risk': '🔴 High',   'action': 'Pseudonymize'},
    {'column': 'full_name',             'pii_type': 'Direct identifier',    'gdpr_category': 'Personal data',             'risk': '🔴 High',   'action': 'Pseudonymize'},
    {'column': 'email',                 'pii_type': 'Direct identifier',    'gdpr_category': 'Personal data',             'risk': '🔴 High',   'action': 'Pseudonymize'},
    {'column': 'ssn',                   'pii_type': 'Direct identifier',    'gdpr_category': 'Personal data',             'risk': '🔴 High',   'action': 'Pseudonymize'},
      
    # Quasi-identifiers — can identify a person when combined
    {'column': 'date_of_birth',         'pii_type': 'Quasi-identifier',     'gdpr_category': 'Personal data',             'risk': '🟡 Medium', 'action': 'Generalize → birth_year (k-anonymity)'},
    {'column': 'zip_code',              'pii_type': 'Quasi identifier',     'gdpr_category': 'Personal data',             'risk': '🟡 Medium', 'action': 'Generalize'},
    {'column': 'gender',                'pii_type': 'Quasi-identifier',     'gdpr_category': 'Personal data',             'risk': '🟡 Medium', 'action': 'Retain — required for bias audit'},
   
    # Sensitive financial data
    {'column': 'annual_income',         'pii_type': 'Financial data',       'gdpr_category': 'Personal data',             'risk': '🟡 Medium', 'action': 'Differential privacy noise'},
    {'column': 'savings_balance',       'pii_type': 'Financial data',       'gdpr_category': 'Personal data',             'risk': '🟡 Medium', 'action': 'Retain — core model feature'},
    {'column': 'debt_to_income',        'pii_type': 'Derived financial',    'gdpr_category': 'Personal data',             'risk': '🟢 Low',    'action': 'Retain — derived ratio'},
    {'column': 'credit_history_months', 'pii_type': 'Derived financial',    'gdpr_category': 'Personal data',             'risk': '🟢 Low',    'action': 'Retain — derived metric'},
    
    #Technical identifiers
    {'column': 'ip_address',            'pii_type': 'Technical identifier', 'gdpr_category': 'Personal data',     'risk': '🟡 Medium', 'action': 'Mask / anonymize'},

    # Free text (potential privacy risk)
    {'column': 'notes', 'pii_type': 'Free text field', 'risk': '🔴 High', 'action': 'Review / redact potential PII'},
    
    # Metadata
    {'column': 'processing_timestamp', 'pii_type': 'Metadata / temporal',  'gdpr_category': 'Personal data',             'risk': '🟢 Low',    'action': 'Define retention policy'},
    {'column': 'loan_approved',             'pii_type': 'Decision output',       'gdpr_category': 'Personal data (Art. 22)',  'risk': '🟡 Medium', 'action': 'Retain — audit target'},
])


# Only show columns that actually exist in the dataframe
pii_existing = pii_classification[pii_classification['column'].isin(df.columns)].copy()

print(f'Total PII fields identified: {len(pii_existing)}')
print(f'High risk fields:   {(pii_existing["risk"] == "🔴 High").sum()}')
print(f'Medium risk fields: {(pii_existing["risk"] == "🟡 Medium").sum()}')
print(f'Low risk fields:    {(pii_existing["risk"] == "🟢 Low").sum()}')
print()
display(pii_existing.set_index('column'))

Total PII fields identified: 15
High risk fields:   5
Medium risk fields: 7
Low risk fields:    3



,pii_type,gdpr_category,risk,action
column,,,,
_id,Direct identifier,Personal data,🔴 High,Pseudonymize
full_name,Direct identifier,Personal data,🔴 High,Pseudonymize
email,Direct identifier,Personal data,🔴 High,Pseudonymize
ssn,Direct identifier,Personal data,🔴 High,Pseudonymize
date_of_birth,Quasi-identifier,Personal data,🟡 Medium,Generalize → birth_year (k-anonymity)
zip_code,Quasi identifier,Personal data,🟡 Medium,Generalize
gender,Quasi-identifier,Personal data,🟡 Medium,Retain — required for bias audit
annual_income,Financial data,Personal data,🟡 Medium,Differential privacy noise
savings_balance,Financial data,Personal data,🟡 Medium,Retain — core model feature


## 2. Privacy Risks Evaluation

The presence of multiple PII fields in the dataset may increase the risk of individual identification due to their sensitive nature, i.e. a person can be easily identified using combinations of attributes such as name, date of birth, and ZIP code.

Since the dataset contains sensitive identifiers such as SSN and email stored directly, this increases the risk of personal data exposure. The first step is to understand whether the company truly needs to store all this information in order to provide credit services to clients. If not, this would violate the GDPR data minimization principle.

Regarding SSN, emails, and full names, it can be observed that there is no hashing, pseudonymization, or encryption applied to these fields that  significantly increases the risk of identification and re-indentification individuals. 

This situation may represent a risk of personal data exposure and potential non-compliance with GDPR data protection principles.

These issues may also affect other GDPR principles, particularly the integrity and confidentiality of personal data, which requires organizations to ensure appropriate security measures when storing sensitive information.

In [6]:
df[existing_pii].head()

,full_name,email,ssn,ip_address,date_of_birth,zip_code
0,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,1986-05-27,90230
1,Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,1999-08-01,10020
2,Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,1982-08-24,90213
3,Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,02/28/1995,90217
4,Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,1960/06/19,90296


In [7]:
print("Number of PII fields:", len(existing_pii))

Number of PII fields: 6


## 3. Pseudonymization Demonstration 
Regarding GDPR principles, organizations should implement technical measures to protect sensitive personal data. The most common measure is pseudonymization. This process consists of substituting direct identifiers with transformed values, reducing the exposure risk of personal data.

Even though the data continues to be used in the analysis, the original identifier stops being directly visible.

In the analyzed dataset, the field of the Social Security Number (SSN) is a highly sensitive identifier. If this information is exposed, it can allow the direct identification of the person.

So, we will demonstrate a simple method of pseudonymization using the hash function, which transforms the original number into a coded identifier.

Although SHA-256 hashing reduces the risk of exposing sensitive identifiers, hashing alone may still be vulnerable to dictionary attacks. In real-world systems, hashing is often combined with salting to strengthen protection against potential re-identification attacks. Salt is when a random value is added before hashing the data. This significantly increases security by making precomputed attacks more difficult. For simplicity, this demonstration uses basic SHA-256 hashing, but a production system should implement salted hashing or other secure pseudonymization techniques.

Although SHA-256 hashing reduces the risk of exposing sensitive identifiers, hashing alone may still be vulnerable to dictionary attacks.
In real-world systems, hashing is often combined with salting to strengthen protection against potential re-identification attacks. Salt is when a random value is added before hashing the data. This significantly increases security by making precomputed attacks more difficult. For simplicity, this demonstration uses basic SHA-256 hashing, but a production system should implement salted hashing or other secure pseudonymization techniques.

In [8]:
def hash_value(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

df["ssn_hash"] = df["ssn"].apply(hash_value)

df[["ssn", "ssn_hash"]].head()

,ssn,ssn_hash
0,427-90-1892,d49cad59b567ffb59422b169add8cf587764aabe1b91f6...
1,992-61-4010,0e1892639ce70228710f0735fa85377403b32b612f398c...
2,833-33-5929,5e93b5041cec1a785d060c243cd2a8f8b80493574aebdd...
3,486-50-5539,3ef30b5e161efd64477f8094eafe1546f79df340fa9ad6...
4,400-91-8156,8c2a4e7afef60f4e60cccf69d18a73b7ccb2e329904802...


We can observe that the original SSN was transformed into a coded identifier.

This process allows us to keep the possibility of data analysis without exposing the original identifier directly.

Despite this, it is important to note that pseudonymization does not completely remove the risk of identification. If there are other identifiers in the data, there may still be a risk of re-identification.

For that reason, pseudonymization must be combined with other data governance practices, such as access limitation, encryption, and data retention policies.

## 4. Data Minimization Evaluation

One of the fundamental principles of GDPR is the principle of Data Minimization (Article 5(1)(c)), organizations must collect and store only the personal data that are strictly necessary to comply with a specific goal.

In this context, the main goal is the evaluation of credit risk. As previously identified, the dataset contains several PII fields. 
The presence of these identifiers raises questions about whether all this information is strictly necessary for credit risk evaluation.

For credit risk evaluation, some financial variables are clearly relevant, such as:  
- annual_income  
- credit_history_months  
- debt_to_income  
- savings_balance

These variables are directly related to the financial capacity of the applicant and are therefore important for credit risk assessment.

However, other fields such as SSN, email, or IP address may not be strictly necessary for evaluating credit risk. If these data are not essential for the decision process, their storage may represent a potential violation of the GDPR Data Minimization principle.

## 5. Governance Gaps
Beyond the privacy risks previously identified, it is important to analyse whether the dataset contains the necessary fields to guarantee compliance with Data Governance good practices and GDPR principles.

An essential part of data governance consists of ensuring that there is sufficient documentation to justify the collection, use, and retention of personal data.

However, while analysing the structure of the dataset, we observed that some important fields needed to ensure transparency and compliance seem to be missing.

Among these fields we can highlight:

- consent_timestamp – to record when the user gave consent for the use of their data  
- processing_purpose – to identify the specific purpose of the data processing  
- retention_until – to define until when the data can be stored  
- data_source – to identify the origin of the data  

The absence of this information may make compliance audits more difficult and make it harder to demonstrate GDPR compliance, especially regarding transparency, purpose limitation, and storage limitation.


In [10]:
expected_governance_fields = [
    "consent_timestamp",
    "processing_purpose",
    "retention_until",
    "data_source"
]

missing_fields = [field for field in expected_governance_fields if field not in df.columns]

print("Missing Governance Fields:")
print(missing_fields)

Missing Governance Fields:
['consent_timestamp', 'processing_purpose', 'retention_until', 'data_source']


## 6. Data Governance Recommendations

Based on the privacy risks and governance gaps identified previously, it is possible to propose some measures to improve the management and protection of personal data in this system. These recommendations are aligned with the principles of the GDPR and with good Data Governance practices.

**1. Implementation of pseudonymization**  
Sensitive data such as SSNs should be protected through pseudonymization or encryption techniques.
This reduces the risk of exposure of personal data in case of unauthorized access.

**2. Definition of data retention policies**  
The organization should define clear data retention policies, specifying how long personal data can be stored and when it should be deleted.

**3. Consent registration**  
It is important to implement a mechanism to record when the user has given consent for the use of their personal data.
This can be done through a field such as consent_timestamp.

**4. Limitation of access to personal data**  
Access to personal data should be restricted to authorized users only.
This can be done through access control mechanisms.

**5. Documentation of the purpose of processing**  
The system should include information about the purpose of collecting and using personal data, for example through a processing_purpose field.
These measures help reduce privacy risks, improve transparency, and ensure greater compliance with the principles of the GDPR.

## 7. Link with Bias Analysis

Although this analysis focused mainly on privacy and data governance aspects, it is also important to consider potential bias in automated credit decision systems.

The bias audit performed in the data science analysis evaluates whether approval decisions differ across demographic groups such as gender or age.

From a governance perspective, monitoring bias is essential to ensure fairness and regulatory compliance. Automated credit scoring systems may produce discriminatory outcomes if they are not properly monitored.

In addition, under the European Union AI Act, credit scoring systems are commonly classified as high-risk AI systems. For this reason, organizations must implement governance mechanisms such as fairness monitoring, regular bias audits, and human oversight of automated decisions.

## Conclusion

This analysis identified several privacy risks in the dataset, mainly related to the presence of multiple personal identifiers such as name, email, SSN, and other sensitive information.

The evaluation also showed that some of these identifiers are stored without protection, which may increase the risk of personal data exposure. To reduce this risk, a pseudonymization approach was demonstrated using hashing techniques.

In addition, the analysis considered the GDPR principle of Data Minimization and highlighted that some personal data may not be strictly necessary for credit risk evaluation.

Some governance gaps were also identified, such as the absence of fields related to consent, processing purpose, and data retention. Implementing proper data governance measures can help improve transparency, reduce privacy risks, and ensure better compliance with GDPR principles.

Finally, since credit scoring systems may be considered high-risk AI systems under the EU AI Act, organizations should also ensure strong governance and oversight mechanisms when using automated decision systems.

In addition, the bias audit conducted in the data science analysis highlights the importance of monitoring potential discrimination in automated credit decisions.

From a governance perspective, fairness assessments should be considered an essential control when deploying AI systems that affect individuals' financial opportunities.

Finally, since credit scoring systems may be considered high-risk AI systems under the EU AI Act, organizations should also ensure strong governance and oversight mechanisms when using automated decision systems.

In addition, the bias audit conducted in the data science analysis highlights the importance of monitoring potential discrimination in automated credit decisions. From a governance perspective, fairness assessments should be considered an essential control when deploying AI systems that affect individuals' financial opportunities.